# Project Overview
This notebook aims to study and explore **memory** and **tools** in LangGraph.

The objective is to understand how LangGraph handles memory management and integrates external tools into its workflow. The notebook demonstrates practical usage examples, allowing us to analyze how agents can store, retrieve, and utilize context while interacting with different tools.

In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.prebuilt import ToolNode, tools_condition
import requests
import os
from langchain_openai import ChatOpenAI
from typing import TypedDict

In [ ]:
load_dotenv(override=True)

In [ ]:
# Explanation: Creamos nuestro primer tool con Landgraph
from langchain_community.utilities import GoogleSerperAPIWrapper
serper = GoogleSerperAPIWrapper()
result = serper.run("¿Cuál es la capital de Guinea Ecuatorial?")
print(result)

In [ ]:
# Explanation: usamos una funcion envolvedorra de Landgraph para convertir cualquier funcion en una Tool
from langchain.agents import Tool
tool_search = Tool(
    name="search",
    func=serper.run,
    description = "Util cuando se trata de buscar informacion por internet"

)


In [ ]:
tool_search.invoke("Cuál es la capital de Guinea Ecuatorial?")

In [ ]:
# Explanation: Ahora hagamos una tool por nuestra cuenta
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"

def push (text:str):
    requests.post(pushover_url, data={"token":pushover_token, "user":pushover_user, "message":text})

In [ ]:
tool_push = Tool(
    name="push",
    func=push,
    description="useful to receive a push notofication"
)

tool_push.invoke("hello, me")

In [ ]:
# Explanation: Pongamos todas las tools en una misma lista
# Explanation: El objetivo es hacer un chat donde ya metamos OpenAI directamente (como en el proyecto anterior)
tools = [tool_search, tool_push]


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory= MemorySaver()

In [ ]:
# Explanation: 1 create a state
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# Explanation: 2 Create a graph
graph_builder = StateGraph(State)

In [ ]:
# Explanation: 3 Create a node
# Explanation: First we choose de llm and bind it with the currents tools
from langgraph import graph


llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    print(state)
    return {"messages": llm_with_tools.invoke(state["messages"])}

# Explanation: Añadimos el nodo al grafo
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))


In [ ]:
# Explanation: 4 Create edges
graph_builder.add_conditional_edges("tools", tools_condition, "chatbot")
graph_builder.add_edge(START, "chatbot")

In [ ]:
# Explanation: 5 compile graph
# Explanation: from langgraph.types import Checkpointer

graph = graph_builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "1"}}

def chat(user_input: str, history):
    try:
        result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
        return result["messages"][-1].content
    except:
# Explanation: Buscar último checkpoint que funcione
        history_checkpoints = list(graph.get_state_history(config))
        for checkpoint in history_checkpoints[1:]:
            try:
                graph.invoke(None, config=checkpoint.config)
# Explanation: Usar directamente checkpoint.config (sin update)
                result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=checkpoint.config)
                return result["messages"][-1].content
            except:
                continue
        return "Error: No se puede recuperar la conversación"

gr.ChatInterface(chat, type="messages").launch()

In [ ]:
graph.get_state(config)

In [ ]:
# Explanation: Most recet first
list(graph.get_state_history(config))

Ahora vamos a alamcenar en sqlite

# Section 3: SQL-based Memory
This section explores LangGraph with **SQL-backed memory**. The agent persists its memory into a SQL database, allowing retrieval across sessions.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

db_path = "memory.db"
conn=sqlite3.connect(db_path, check_same_thread=False)
sql_memory = SqliteSaver(conn)

In [ ]:
# Explanation: 1 & 2
graph_builder = StateGraph(State)

# Explanation: 3 node
def chatbot(state: State):
    print(state)
    return {"messages": llm_with_tools.invoke(state["messages"])}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

# Explanation: 4 Edges
graph_builder.add_conditional_edges("tools", tools_condition, "chatbot")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

# Explanation: 5 compile graph
graph = graph_builder.compile(checkpointer=sql_memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "3"}}

def chat(user_input: str, history):
    try:
        result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
        return result["messages"][-1].content
    except:
# Explanation: Buscar último checkpoint que funcione
        history_checkpoints = list(graph.get_state_history(config))
        for checkpoint in history_checkpoints[1:]:
            try:
                graph.invoke(None, config=checkpoint.config)
# Explanation: Usar directamente checkpoint.config (sin update)
                result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=checkpoint.config)
                return result["messages"][-1].content
            except:
                continue
        return "Error: No se puede recuperar la conversación"

gr.ChatInterface(chat, type="messages").launch()